# BirdCLEF+ 2026

Uses baseline from [tonylica/birdclef-2026-model](https://www.kaggle.com/datasets/tonylica/birdclef-2026-model).

## Score:.892

In [ ]:
import os
import re
import time
import glob
import warnings
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T
import torchvision.models as tv_models
from scipy.ndimage import zoom

warnings.filterwarnings("ignore")

START = time.time()
print(f"PyTorch {torch.__version__}")

EXP_ID = 5
W_OWN = 0.15

OWN_CKPT_CANDIDATES = [
    "models/best.pt",
    "/kaggle/working/models/best.pt",
    str(Path.cwd() / "models" / "best.pt"),
]
if os.path.exists("/kaggle/input"):
    for d in Path("/kaggle/input").iterdir():
        p = d / "best.pt"
        if p.exists():
            OWN_CKPT_CANDIDATES.insert(0, str(p))
            break
OWN_CKPT = next((p for p in OWN_CKPT_CANDIDATES if os.path.exists(p)), None)

@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    max_workers: int = 4

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)

cfg = Config()

KAGGLE_DATA = "/kaggle/input/competitions/birdclef-2026"
LOCAL_CANDIDATES = [
    Path.cwd() / "birdclef-2026",
    Path.cwd() / ".." / "birdclef-2026",
]
DATA_ROOT = KAGGLE_DATA if os.path.exists(KAGGLE_DATA) else None
if DATA_ROOT is None:
    for p in LOCAL_CANDIDATES:
        if p and p.resolve().exists() and (p / "sample_submission.csv").exists():
            DATA_ROOT = str(p.resolve())
            break
if DATA_ROOT is None:
    DATA_ROOT = str(Path.cwd())
DATA_ROOT = os.path.normpath(DATA_ROOT)
print(f"DATA_ROOT: {DATA_ROOT}")

TEST_DIR = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")

CKPT_CANDIDATES = [
    "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB872.pt",
    "/kaggle/input/tonylica-birdclef-2026-model/LB872.pt",
    str(Path(DATA_ROOT) / "models" / "LB872.pt"),
    str(Path.cwd() / "models" / "LB872.pt"),
]
if not any(os.path.exists(p) for p in CKPT_CANDIDATES) and os.path.exists("/kaggle/input"):
    for d in Path("/kaggle/input").iterdir():
        p872 = d / "LB872.pt"
        if p872.exists():
            CKPT_CANDIDATES.insert(0, str(p872))
            break
FINETUNED_CKPT = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)
BASELINE_CKPT = FINETUNED_CKPT.replace("LB872", "LB862") if FINETUNED_CKPT else None

if not FINETUNED_CKPT or not os.path.exists(BASELINE_CKPT):
    raise FileNotFoundError(
        "Checkpoints not found. On Kaggle: add dataset tonylica/birdclef-2026-model. "
        "Locally: put LB872.pt and LB862.pt in ./models/ or birdclef-2026/models/"
    )

BLEND_OPTIONS = [
    ("finetuned_only", {"mode": "prob", "ft": 1.0, "base": 0.0}),
    ("baseline_only", {"mode": "prob", "ft": 0.0, "base": 1.0}),
    ("prob_ft80_base20", {"mode": "prob", "ft": 0.8, "base": 0.2}),
    ("prob_ft70_base30", {"mode": "prob", "ft": 0.7, "base": 0.3}),
    ("prob_ft50_base50", {"mode": "prob", "ft": 0.5, "base": 0.5}),
    ("logit_ft80_base20", {"mode": "logit", "ft": 0.8, "base": 0.2}),
    ("logit_ft70_base30", {"mode": "logit", "ft": 0.7, "base": 0.3}),
    ("logit_ft50_base50", {"mode": "logit", "ft": 0.5, "base": 0.5}),
]
assert 0 <= EXP_ID < len(BLEND_OPTIONS)
SELECTED_NAME, SELECTED_SPEC = BLEND_OPTIONS[EXP_ID]
print(f"Blend: EXP_ID={EXP_ID} -> {SELECTED_NAME}")

sample_sub_path = os.path.join(DATA_ROOT, "sample_submission.csv")
if not os.path.exists(sample_sub_path):
    raise FileNotFoundError(f"Competition data not found. Add birdclef-2026. Looked at: {sample_sub_path}")
sample_sub = pd.read_csv(sample_sub_path)
SPECIES = list(sample_sub.columns[1:])
cfg.num_classes = len(SPECIES)
print(f"Species: {len(SPECIES)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {"clipwise_logit": clipwise_logit, "clipwise_prob": torch.sigmoid(clipwise_logit)}


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
            features_only=False, global_pool="", num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.mel(waveforms), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.db(mel), nan=-80.0, posinf=0.0, neginf=-80.0)
        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


def safe_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_model(ckpt_path, tag):
    ckpt = safe_load(ckpt_path)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model = SEDModel(cfg)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    epoch = ckpt.get("epoch", "?") if isinstance(ckpt, dict) else "?"
    auc = ckpt.get("metrics", {}).get("macro_auc", "?") if isinstance(ckpt, dict) else "?"
    print(f"{tag}: loaded | epoch={epoch} | val_auc={auc}")
    return model


baseline_model = load_model(BASELINE_CKPT, "baseline")
finetuned_model = load_model(FINETUNED_CKPT, "finetuned")
mel_transform = MelSpectrogramTransform(cfg).to(device).eval()

class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=False):
        super().__init__()
        try:
            w = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = tv_models.efficientnet_b0(weights=w)
        except Exception:
            self.backbone = tv_models.efficientnet_b0(weights=None)
        self.backbone.classifier[1] = nn.Linear(self.backbone.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

def audio_to_mel_own(y, sr=32000, n_mels=128, n_fft=2048, hop_len=512, img_size=(128, 128)):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_len, fmin=0, fmax=16000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    h, w = mel_db.shape
    mel_resized = zoom(mel_db, (img_size[0] / h, img_size[1] / w), order=1)
    return mel_resized.astype(np.float32)

own_model = None
if OWN_CKPT and W_OWN > 0:
    ckpt = safe_load(OWN_CKPT)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    own_model = EfficientNetClassifier(len(SPECIES), pretrained=False)
    own_model.load_state_dict(state, strict=True)
    own_model.to(device).eval()
    print(f"own_model: loaded from {OWN_CKPT}")
else:
    W_OWN = 0.0

In [ ]:
row_pattern = re.compile(r"^(.*)_(\d+)$")

def parse_row_id(rid):
    m = row_pattern.match(str(rid))
    return (m.group(1), int(m.group(2))) if m else (None, None)

expected_ids = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem:
        expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
if not test_files:
    for stem in sorted(expected_by_stem):
        p = os.path.join(TRAIN_SC_DIR, f"{stem}.ogg")
        if os.path.exists(p):
            test_files.append(p)
    if test_files:
        print(f"No test .ogg; using {len(test_files)} train_soundscapes fallback.")
RUN_VALIDATION = len(test_files) == 0 and os.path.isdir(TRAIN_SC_DIR)
if RUN_VALIDATION:
    train_ogg = sorted(glob.glob(os.path.join(TRAIN_SC_DIR, "*.ogg")))
    if train_ogg:
        test_files = train_ogg[:5]
        print(f"Validation mode: running on {len(test_files)} train_soundscapes.")
if not test_files:
    print("No soundscape files found. Submission will have placeholder predictions.")

def load_soundscape(path):
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32), Path(path).stem

print(f"Soundscape files: {len(test_files)}")
print("Loading audio...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
    results = list(ex.map(load_soundscape, test_files))
print(f"Loaded {len(results)} files in {time.time()-t0:.1f}s")

In [ ]:
def probs_from_blend(base_logits, ft_logits, base_probs, ft_probs, mode, ft_w, base_w):
    if mode == "prob":
        probs = ft_w * ft_probs + base_w * base_probs
    else:
        logits = ft_w * ft_logits + base_w * base_logits
        probs = torch.sigmoid(logits)
    return torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)

def merge_windows_to_chunks(probs_windows, starts, n_chunks):
    merged = np.zeros((n_chunks, len(SPECIES)), dtype=np.float64)
    counts = np.zeros((n_chunks, 1), dtype=np.float64)
    for j, s in enumerate(starts):
        i_lo = max(0, s // CHUNK)
        i_hi = min(n_chunks - 1, (s + CHUNK - 1) // CHUNK)
        for i in range(i_lo, i_hi + 1):
            merged[i] += probs_windows[j]
            counts[i] += 1
    return (merged / np.maximum(counts, 1)).astype(np.float32)

CHUNK = cfg.chunk_samples
STRIDE = CHUNK // 2
TTA_SHIFT = CHUNK // 4
all_row_ids, all_preds = [], []

print("Running inference (50% overlap + 2x TTA; keep this light to avoid Kaggle notebook timeout)...")
t0 = time.time()
with torch.no_grad():
    for audio, stem in results:
        if stem in expected_by_stem:
            n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, len(audio) // CHUNK)
        padded_len = n_chunks * CHUNK
        audio = np.pad(audio, (0, max(0, padded_len - len(audio))))[:padded_len]

        starts = list(range(0, len(audio) - CHUNK + 1, STRIDE))
        windows = np.stack([audio[s:s + CHUNK] for s in starts])
        windows_shift = np.roll(windows, TTA_SHIFT, axis=1)

        def run_windows(w):
            t = torch.from_numpy(w).float().to(device)
            mel = mel_transform(t)
            ob = baseline_model(mel)
            of = finetuned_model(mel)
            p = probs_from_blend(
                ob["clipwise_logit"], of["clipwise_logit"],
                torch.nan_to_num(ob["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0),
                torch.nan_to_num(of["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0),
                SELECTED_SPEC["mode"], SELECTED_SPEC["ft"], SELECTED_SPEC["base"],
            ).cpu().numpy()
            return p

        def run_own_windows(w):
            mels = np.stack([audio_to_mel_own(w[i]) for i in range(len(w))])
            mels = (mels - mels.min(axis=(1,2), keepdims=True)) / (mels.max(axis=(1,2), keepdims=True) - mels.min(axis=(1,2), keepdims=True) + 1e-8)
            x = np.stack([mels, mels, mels], axis=1)
            t = torch.from_numpy(x).float().to(device)
            logits = own_model(t)
            return torch.sigmoid(logits).cpu().numpy()

        probs_orig = run_windows(windows)
        probs_shift = run_windows(windows_shift)
        probs_w = (probs_orig + probs_shift) / 2.0
        probs = merge_windows_to_chunks(probs_w, starts, n_chunks)

        if own_model is not None:
            probs_own_orig = run_own_windows(windows)
            probs_own_shift = run_own_windows(windows_shift)
            probs_own_w = (probs_own_orig + probs_own_shift) / 2.0
            probs_own = merge_windows_to_chunks(probs_own_w, starts, n_chunks)
            probs = probs * (1.0 - W_OWN) + probs_own * W_OWN

        if n_chunks > 4:
            sharp = probs ** 1.5
            w = np.array([0.05, 0.15, 0.60, 0.15, 0.05])
            pad = np.pad(sharp, ((2, 2), (0, 0)), mode="edge")
            smooth = w[0]*pad[:-4] + w[1]*pad[1:-3] + w[2]*pad[2:-2] + w[3]*pad[3:-1] + w[4]*pad[4:]
            probs = smooth ** (1/1.5)
        elif n_chunks > 2:
            w = np.array([0.20, 0.60, 0.20])
            pad = np.pad(probs, ((1, 1), (0, 0)), mode="edge")
            probs = w[0]*pad[:-2] + w[1]*pad[1:-1] + w[2]*pad[2:]

        file_max = np.max(probs, axis=0, keepdims=True) * 0.05
        probs = probs + file_max

        if stem in expected_by_stem:
            valid = [(e // int(cfg.chunk_duration)) - 1 for e in expected_by_stem[stem]]
            valid = [i for i in valid if 0 <= i < n_chunks]
        else:
            valid = list(range(n_chunks))
        for i in valid:
            all_row_ids.append(f"{stem}_{(i+1)*int(cfg.chunk_duration)}")
            all_preds.append(probs[i])

print(f"Inference done in {time.time()-t0:.1f}s")

In [ ]:
if all_preds:
    pred_df = pd.DataFrame(np.asarray(all_preds, dtype=np.float32), columns=SPECIES, index=all_row_ids)
    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]
else:
    pred_df = pd.DataFrame(np.zeros((0, len(SPECIES)), dtype=np.float32), columns=SPECIES, index=pd.Index([], name="row_id"))

SC_LABELS_PATH = os.path.join(DATA_ROOT, "train_soundscapes_labels.csv")
if os.path.exists(SC_LABELS_PATH) and len(pred_df) > 0:
    sc_df = pd.read_csv(SC_LABELS_PATH)
    species_counts = {s: 0 for s in SPECIES}
    for row in sc_df["primary_label"].dropna():
        for sp in str(row).split(";"):
            sp = sp.strip()
            if sp in species_counts:
                species_counts[sp] += 1
    total_seg = max(len(sc_df), 1)
    base_rates = np.array([species_counts[s] / total_seg for s in SPECIES], dtype=np.float32)
    prior_mask = (base_rates == 0).astype(np.float32)
    suppression = 1.0 - 0.15 * prior_mask
    pred_df[SPECIES] = pred_df[SPECIES].values * suppression
    n_suppressed = int(prior_mask.sum())
    if n_suppressed > 0:
        print(f"Soundscape prior: {n_suppressed} species gently suppressed")

pred_df[SPECIES] = pred_df[SPECIES].clip(lower=0)

if RUN_VALIDATION and len(pred_df) > 0:
    pred_df.to_csv("validation_output.csv")
    print(f"Validation: saved validation_output.csv ({len(pred_df)} rows)")

missing = [rid for rid in expected_ids if rid not in pred_df.index]
if missing:
    pred_df = pd.concat([pred_df, pd.DataFrame(
        np.zeros((len(missing), len(SPECIES)), dtype=np.float32),
        columns=SPECIES, index=pd.Index(missing, name="row_id")
    )])

submission = pred_df.loc[expected_ids].reset_index()
if submission.columns[0] != "row_id":
    submission = submission.rename(columns={submission.columns[0]: "row_id"})
submission = submission[["row_id"] + SPECIES]
submission.to_csv("submission.csv", index=False)

print(f"Saved submission.csv | shape={submission.shape}")
print(f"Total time: {time.time()-START:.1f}s")
submission.head()